### **Imports:**
Hieronder zijn alle de packages die ik nodig heb om mijn recomonder model te bouwen.

In [17]:
import re
import nltk
from nltk.corpus import stopwords
import pandas as pd
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### **NLP:**
Hieronder pas ik NLP toe op mijn dataset om het goed en lees baar te maken voor de computer (models).

In [18]:
dataset = pd.read_csv('VKM_dataset_cleaned.csv')

#De kolom combined_text maken door alle tekstkolommen samen te voegen die nodig zijn voor elke rij. 
dataset['combined_text'] = (
    dataset['name'].fillna('') + ' ' +
    dataset['shortdescription'].fillna('') + ' ' +
    dataset['description'].fillna('') + ' ' +
    dataset['content'].fillna('') + ' ' +
    dataset['learningoutcomes'].fillna('') + ' ' +
    dataset['module_tags'].fillna('')
)

#Stopword functie aanroepen
stop_words = set(stopwords.words('dutch'))
preprocessed_texts = []

#Voor elke rij in combined_text kolom NLP uitvoeren.
for text in dataset['combined_text']:
    #Tekst lowercase maken
    text_lower = text.lower()
    #alle niet relevante nummers uit de tekst halen.
    text_no_numbers = re.sub(r'\d+', '', text_lower)
    #alle leestekens uit de tekst halen.
    text_no_punct = re.sub(r'[^\w\s]', '', text_no_numbers)
    
    #alle woorden appart maken om de stop woorden eruit te halen.
    tokens = []
    for sentence in sent_tokenize(text_no_punct):
        words = word_tokenize(sentence)
        filtered = [w for w in words if w not in stop_words]
        tokens.extend(filtered)
    
    #Tokanization weer terugdraaien, dus de ipv apparte woorden weer terug naar een gehele zin. 
    preprocessed_texts.append(' '.join(tokens))

### **Content-based Recommender model:**
Nadat ik NLP heb uitgevoerd heb ik hieronder de contend-based recommender gebouwed met de model if-idf. 

In [19]:
vectorizer = TfidfVectorizer()
#Bekijkt elke document(rij) en geeft van elke rij de unieke waarden een score.
tfidf_matrix = vectorizer.fit_transform(preprocessed_texts)
#Hier maak ik een nieuwe kolom om alle if-idf scores van 1 rij op te sommen en in de nieuwe kolom te stoppen. 
dataset['tfidf_score'] = tfidf_matrix.sum(axis=1).A1 

### **Hybride Model:**
Hieronder pas ik de hybride model toe op mijn dataset die een top 5 keuzemodules geeft op basis van de tf-idf score, interest en popularity score. 

In [30]:
scaler = MinMaxScaler()

#Hier maak ik 3 nieuwe kolommen waarvan de interest, popularity en tf-idf score worden gescaled tussen -1 en 1 voor elke rij. 
dataset['interest_match_norm'] = scaler.fit_transform(dataset['interests_match_score'].values.reshape(-1,1))
dataset['popularity_score_norm'] = scaler.fit_transform(dataset['popularity_score'].values.reshape(-1,1))
dataset['tfidf_score_norm'] = scaler.fit_transform(dataset['tfidf_score'].values.reshape(-1,1))
#Hier ga ik een nieuwe kolom maken waarvan ik bepaal hoe zwaar elke onderdeel voor elke rij meeteld en dan tel ik het bij elkaar op.
dataset['hybrid_score'] = (0.5 * dataset['interest_match_norm'] + 0.1 * dataset['popularity_score_norm'] + 0.4 * dataset['tfidf_score_norm'])

#Top 5 uitprinten, dus van de hybride_score kolom de 5 keuzemodules waarvan de score het hoogste is laat ik hierzo zien.
top5_hybrid = dataset.sort_values(by='hybrid_score', ascending=False).head(5)

print('top 5 keuzemodules:\n', top5_hybrid[['name']])

top 5 keuzemodules:
                                                   name
198                                         stopmotion
182  multdisciplinair samenwerken in een beroepscon...
118                   robotic ai interfaces - optie 2*
51          act for change together nlqf6 30 + 15 ects
112                                    robot challenge


### **student Input:**
Hieronder voer ik een student input toe waarna ik de tekst van de student input process om te gebruiken voor het model.

In [33]:
#Input van een student.
student_input = "ik vindt het leuk om te coderen, gamen en sporten."

#NLP uitvoeren voor de input van de student.
student_profile_lower_text = student_input.lower()
student_profile_no_numbers = re.sub(r'\d+', '', student_profile_lower_text)
student_profile_no_punct = re.sub(r'[^\w\s]', '', student_profile_no_numbers)
tokens = []
for sentence in sent_tokenize(student_profile_no_punct):
    words = word_tokenize(sentence)
    filtered = [w for w in words if w not in stop_words]
    tokens.extend(filtered)

#De woorden weer tot een zin maken.
student_text = ' '.join(tokens)

#De input krijgt tf-idf score op basis van de eerder getrainde dataset en dat is dus wat er in de tfidfmatrix zit. dus nieuwe woorden krijg score van 0 en woorden die in de dataset komen met deze scoren krijgen een hoger score. 
student_vector = vectorizer.transform([student_text])

### **cosine** 
gebruik maken van cosin om de tfdf dus hoevaak iets voorkomt in de dataset te vergelijken met de student vector om dan een top 5 te genereren op de basis van de cosin similarity.

In [34]:
#hier wordt gekeken welke woorden in student_vector en tf-idf_matrix heel vergelijkbaar met elkaar zijn en die krijgt een score tussen -1 en 1. 
similarities = cosine_similarity(student_vector, tfidf_matrix)

#Je gaat hier basically van arrays in één array naar 1 array toe. [[1], [2]] -> [1, 2]
similarities = similarities.flatten()
#Pakt de top 5 modules
top_indices = similarities.argsort()[::-1][:5]  

#laat alle kolommen van de top 5 zien.
recommended_modules = dataset.iloc[top_indices]
recommended_modules.head()

,id,name,shortdescription,description,content,studycredit,location,contact_id,level,learningoutcomes,...,start_date,status,combined_text,tfidf_score,interest_match_norm,popularity_score_norm,tfidf_score_norm,hybrid_score,cosine_sim,cosine_sim_norm
114,301,serious gaming - optie 2*,"serious gaming, revalidatie, sporten, game des...",student leert in een interdisciplinaire leerom...,student leert in een interdisciplinaire leerom...,15,Den Bosch,101,NLQF5,je ontwerpt en realiseert in teamverband een a...,...,2025-09-10,definitief,"serious gaming - optie 2* serious gaming, reva...",5.437408,0.640625,0.502041,0.395716,0.257472,0.000000,0.000000
138,325,makerspace,"makerspace, uitvinden, engineering, design thi...",in deze module bedenken studenten een oplossin...,in deze module bedenken studenten een oplossin...,15,Breda,101,NLQF6,de student vindt een oplossing voor een groot ...,...,2025-10-29,definitief,"makerspace makerspace, uitvinden, engineering,...",6.223427,0.375000,0.585714,0.471998,0.227971,0.000000,0.000000
199,386,graphic novel,"creative writing, storytelling skills, genres,...",een graphic novel is een strip met een meer vo...,een graphic novel is een strip met een meer vo...,30,Breda,120,NLQF6,you will establish a practice of writing:\r\n\...,...,2025-11-07,definitief,"graphic novel creative writing, storytelling s...",9.470321,0.796875,0.257143,0.787103,0.342510,0.000000,0.000000
208,395,avans innovative studio senior,ook heb je de mogelijkheid om met creatieve on...,ook heb je de mogelijkheid om met creatieve on...,ook heb je de mogelijkheid om met creatieve on...,30,Breda,92,NLQF6,de student demonstreert persoonlijke groei op ...,...,2025-11-29,definitief,avans innovative studio senior ook heb je de m...,4.770430,0.000000,0.193878,0.330987,0.396499,0.086494,0.621828
207,394,avans innovative studio junior,"persoonlijke ontwikkeling, interdisciplinair, ...",je leert hoe welke waarde jouw werk heeft. hoe...,je leert hoe welke waarde jouw werk heeft. hoe...,15,Breda,92,NLQF6,de student demonstreert persoonlijke groei op ...,...,2025-09-01,definitief,avans innovative studio junior persoonlijke on...,5.665821,0.671875,0.589796,0.417883,0.276931,0.000000,0.000000


In [35]:
#alle woorden (features) die vectorizer heeft geleerd van de preprocessed_texts ofwel de combined_kolom van mijn dataset kolommen die ik heb gebruikt.
feature_names = vectorizer.get_feature_names_out()

for i in top_indices:
    #Haalt alle tf-idf scores en zet het in een array.
    doc_vector = tfidf_matrix[i].toarray()[0]
    #Zet de woorden bij de juiste tf-idf scoren.
    word_scores = dict(zip(feature_names, doc_vector))
    #Haalt de 5 hoogste tf-idf scores met hen woord.  
    top_words = sorted(word_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    #naam module
    print(f"Module: {dataset.iloc[i]['name']}")
    #print de top 5 woorden van elke rij (keuzemodule)
    print("Waarom passend? Belangrijkste woorden:", [w[0] for w in top_words])
    print()

Module: serious gaming - optie 2*
Waarom passend? Belangrijkste woorden: ['gebruiker', 'aanzet', 'serious', 'game', 'gaming']

Module: makerspace
Waarom passend? Belangrijkste woorden: ['makerspace', 'oplossing', 'probleem', 'thinking', 'aanleren']

Module: graphic novel
Waarom passend? Belangrijkste woorden: ['novel', 'échte', 'graphic', 'verhaal', 'karakter']

Module: avans innovative studio senior
Waarom passend? Belangrijkste woorden: ['buiten', 'onderwijsmuren', 'sparren', 'ervaringen', 'ondernemers']

Module: avans innovative studio junior
Waarom passend? Belangrijkste woorden: ['kijker', 'valorisatie', 'persoonlijke', 'interdisciplinair', 'prototyping']



### **Hybride Model+**
In deze hybride model voeg ik de similarity score ook toe.

In [36]:
dataset['cosine_sim'] = similarities

dataset['cosine_sim_norm'] = scaler.fit_transform(dataset['cosine_sim'].values.reshape(-1,1))

dataset['hybrid_score'] = (0.5 * dataset['cosine_sim_norm'] + 0.2 * dataset['interest_match_norm'] + 0.2 * dataset['tfidf_score_norm'] + 0.1 * dataset['popularity_score_norm'])
top5_hybrid = dataset.sort_values(by='hybrid_score', ascending=False).head(5)

print('top 5 keuzemodules:\n', top5_hybrid[['name']])


top 5 keuzemodules:
                                  name
114         serious gaming - optie 2*
199                     graphic novel
138                        makerspace
118  robotic ai interfaces - optie 2*
198                        stopmotion
